In [1]:
import sys
import yaml
import argparse
import numpy as np
import igraph as ig
import networkx as nx
import matplotlib.pyplot as plt
from qlgates.config import Config
from qlgates.run_dynamics import propagate_state, build_unitary, bell_state
from qlgates.qlgraphs import qldit, cart_qldit, compute_eigen_decomposition
from qlgates.gates import get_Vg
from core.contraction import minimal_quotient
from core.graph_generation import generate_quantum_like_bit

In [4]:
print(sys.executable)   # should show your conda env path, not system python

"""Load YAML and merge into Config dataclass."""
with open("../configs/contraction.yaml") as f:
    overrides = yaml.safe_load(f)  # plain dict from YAML

# These are computed in __post_init__, never let YAML set them
overrides.pop("l", None)
overrides.pop("lp", None)

# Create Config instance with merged parameters
cfg = Config(**overrides)

/Users/sahadebadrita/opt/anaconda3/envs/graphs/bin/python


In [20]:
print(cfg)

Config(debug=False, n=32, k=20, d=1, l=5, lp=4, coupling=True, periodic=False, full=False, NQL=3, CartPdt=True, model='transverse', timesteps=100, deltat=0.1, J=-1.0, h=0.2)


In [28]:
#Checking Will's package
qlbit_1p, info_1p = generate_quantum_like_bit(cfg.n,cfg.k,cfg.l)
qlbit_2p, info_2p = generate_quantum_like_bit(cfg.n,cfg.k,cfg.lp)
qlbit_3p, info_3p = generate_quantum_like_bit(cfg.n,cfg.k,cfg.l)

In [29]:
qlbit_1pmin,info_1pmin = minimal_quotient((qlbit_1p,info_1p))
qlbit_2pmin,info_2pmin = minimal_quotient((qlbit_2p,info_2p))
qlbit_3pmin,info_3pmin = minimal_quotient((qlbit_3p,info_3p))

In [30]:
min_cartpdt12 = cart_qldit(qlbit_2pmin,qlbit_3pmin)
min_cartpdt123 = cart_qldit(qlbit_1pmin,min_cartpdt12)
print(min_cartpdt123.shape)

(8, 8)


In [31]:
e_min_cartpdt123, v_min_cartpdt123 = np.linalg.eigh(min_cartpdt123)
print(e_min_cartpdt123[-5:])

[56. 64. 64. 66. 74.]


Test single QL-bit states in min rep

In [32]:
e_qlbit1_min, v_qlbit1_min = np.linalg.eigh(qlbit_1pmin)
print(e_qlbit1_min)
print(v_qlbit1_min)

[15. 25.]
[[-0.70710678  0.70710678]
 [ 0.70710678  0.70710678]]


Check single QL-bit gates

In [33]:
H = get_Vg('H',theta=None,U=None)
X = get_Vg('x',theta=None,U=None)

Ux = H @ X @ H

print(v_qlbit1_min)
print('Ux @ v_qlbit1_min:')
print(Ux @ v_qlbit1_min)
print('Ux @ qlbit_1pmin @ Ux.T.conj():')
print(Ux @ qlbit_1pmin @ Ux.T.conj())

[[-0.70710678  0.70710678]
 [ 0.70710678  0.70710678]]
Ux @ v_qlbit1_min:
[[-0.70710678+0.j  0.70710678+0.j]
 [-0.70710678+0.j -0.70710678+0.j]]
Ux @ qlbit_1pmin @ Ux.T.conj():
[[20.+0.j -5.+0.j]
 [-5.+0.j 20.+0.j]]


In [34]:
H = get_Vg('H',theta=None,U=None)
Z = get_Vg('z',theta=None,U=None)

Uz = H @ Z @ H

print(v_qlbit1_min)
print('Uz @ v_qlbit1_min:')
print(Uz @ v_qlbit1_min)
print('Uz @ qlbit_1pmin @ Uz.T.conj():')
print(Uz @ qlbit_1pmin @ Uz.T.conj())

[[-0.70710678  0.70710678]
 [ 0.70710678  0.70710678]]
Uz @ v_qlbit1_min:
[[ 0.70710678+0.j  0.70710678+0.j]
 [-0.70710678+0.j  0.70710678+0.j]]
Uz @ qlbit_1pmin @ Uz.T.conj():
[[20.+0.j  5.+0.j]
 [ 5.+0.j 20.+0.j]]


In [35]:
H = get_Vg('H',theta=None,U=None)
RX = get_Vg('Rx',theta=1.57,U=None)

URx = H @ RX @ H
print(URx @ v_qlbit1_min)
print(URx @ qlbit_1pmin @ URx.T.conj())

[[-0.50019904+0.49980088j  0.50019904-0.49980088j]
 [ 0.50019904+0.49980088j  0.50019904+0.49980088j]]
[[2.00000000e+01+0.j         3.98163355e-03-4.99999841j]
 [3.98163355e-03+4.99999841j 2.00000000e+01+0.j        ]]


Basis change for 2 ql-bits

In [36]:
e_qlbit2_min, v_qlbit2_min = np.linalg.eigh(qlbit_2pmin)
print(e_qlbit2_min)
print(v_qlbit2_min)

[16. 24.]
[[-0.70710678  0.70710678]
 [ 0.70710678  0.70710678]]


In [37]:
e_min_cartpdt12, v_min_cartpdt12 = np.linalg.eigh(min_cartpdt12)

UCB2 = np.kron(H, H)

print(e_min_cartpdt12)
print(v_min_cartpdt12)
print(np.real(UCB2 @ v_min_cartpdt12))

[31. 39. 41. 49.]
[[ 0.5 -0.5  0.5 -0.5]
 [-0.5  0.5  0.5 -0.5]
 [-0.5 -0.5 -0.5 -0.5]
 [ 0.5  0.5 -0.5 -0.5]]
[[-3.33066907e-16  1.33226763e-15  2.38697950e-15 -1.00000000e+00]
 [-3.33066907e-16 -1.00000000e+00 -8.10462808e-15 -1.27675648e-15]
 [ 5.55111512e-16 -7.88258347e-15  1.00000000e+00  2.38697950e-15]
 [ 1.00000000e+00 -2.77555756e-16 -6.38378239e-16 -5.55111512e-17]]


In [38]:
CNOT = np.array([
    [1, 0, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0],
    [0, 1, 0, 0]
])

In [40]:
UCNOT = UCB2 @ CNOT @ UCB2
print((UCNOT @ v_min_cartpdt12).real)


[[ 0.5 -0.5  0.5 -0.5]
 [-0.5  0.5  0.5 -0.5]
 [ 0.5  0.5 -0.5 -0.5]
 [-0.5 -0.5 -0.5 -0.5]]


In [ ]:
print((UCNOT @ v_min_cartpdt12).real - v_min_cartpdt12.real)